In [1]:
# import

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Loading cleaned csv data

df = pd.read_csv('E:/Git/blore-house-data-with-gradient-boost-and-XGBoost/clean_data.csv')
df.head()

,area,location,bhk,bath,balcony,parking,furnishing,property_type,age,price
0,2065,1,2,3,0,1,2,3,3,17280000
1,1539,0,3,1,0,1,1,2,8,9410000
2,2048,1,3,1,2,0,2,3,10,20300000
3,1233,6,3,2,1,2,3,1,12,9060000
4,1618,0,2,3,1,1,1,1,1,10320000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 981 entries, 0 to 980
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   area           981 non-null    int64
 1   location       981 non-null    int64
 2   bhk            981 non-null    int64
 3   bath           981 non-null    int64
 4   balcony        981 non-null    int64
 5   parking        981 non-null    int64
 6   furnishing     981 non-null    int64
 7   property_type  981 non-null    int64
 8   age            981 non-null    int64
 9   price          981 non-null    int64
dtypes: int64(10)
memory usage: 76.8 KB


### Model Creation

In [4]:
# defining x and y

x = df.drop(columns='price')
y = np.log1p(df['price'])
x.shape

(981, 9)

In [5]:
# Train Test split

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.30, random_state=42)

In [6]:
# model

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score


gbr = GradientBoostingRegressor()

scores1 = cross_val_score(
    gbr,
    x_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("CV Scores:", scores1)
print("Mean CV Score:", scores1.mean())

gbr.fit(x_train, y_train)


y_pred_log1 = gbr.predict(x_test)
y_pred_actual1 = np.expm1(y_pred_log1)
y_test_actual1 = np.expm1(y_test)



CV Scores: [0.60983551 0.50404189 0.5317769  0.63689373 0.57072987]
Mean CV Score: 0.5706555823611342


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test_actual1, y_pred_actual1))
print("MSE:", mean_squared_error(y_test_actual1, y_pred_actual1))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual1, y_pred_actual1)))
print("R2 Score:", r2_score(y_test_actual1, y_pred_actual1))

MAE: 2711771.674477854
MSE: 11200266650739.82
RMSE: 3346679.9444733015
R2 Score: 0.5300178937281732


In [8]:
# XGBoost Model
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

xgb = XGBRegressor(n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42)

scores2 = cross_val_score(
    xgb,
    x_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("CV Scores:", scores2)
print("Mean CV Score:", scores2.mean())

xgb.fit(x_train, y_train)


y_pred_log2 = xgb.predict(x_test)

y_pred_actual2 = np.expm1(y_pred_log2)
y_test_actual2 = np.expm1(y_test)

CV Scores: [0.59341851 0.45441549 0.49579725 0.55859769 0.56271061]
Mean CV Score: 0.5329879100376693


In [9]:
print("MAE:", mean_absolute_error(y_test_actual2, y_pred_actual2))
print("MSE:", mean_squared_error(y_test_actual2, y_pred_actual2))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual2, y_pred_actual2)))
print("R2 Score:", r2_score(y_test_actual2, y_pred_actual2))

MAE: 2714136.8677966096
MSE: 11765108925413.43
RMSE: 3430030.455464416
R2 Score: 0.5063161578462925


#### Hyperparameter Tuning

In [10]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [100, 200, 300, 400, 500, 600, 700],
    'learning_rate': [0.001, 0.01, 0.05, 0.1],
    'max_depth': [1, 2, 3, 4, 5, 6],
    'subsample': [0.7, 0.8, 1.0]
}

grid = GridSearchCV(
    estimator=GradientBoostingRegressor(),
    param_grid=params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)

best_model1 = grid.best_estimator_

In [11]:
y_pred_log3 = best_model1.predict(x_test)

y_pred_actual3 = np.expm1(y_pred_log3)
y_test_actual3 = np.expm1(y_test)

In [12]:
print("MAE:", mean_absolute_error(y_test_actual3, y_pred_actual3))
print("MSE:", mean_squared_error(y_test_actual3, y_pred_actual3))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual3, y_pred_actual3)))
print("R2 Score:", r2_score(y_test_actual3, y_pred_actual3))

MAE: 2657548.250613845
MSE: 10644094674621.156
RMSE: 3262528.877208776
R2 Score: 0.5533558092382783


In [13]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [100, 200, 300, 400, 500, 600],
    'learning_rate': [0.001, 0.01, 0.05, 0.1],
    'max_depth': [1, 2, 3, 4, 5],
    'subsample': [0.7, 0.8, 1.0]
}

grid = GridSearchCV(
    estimator=XGBRegressor(),
    param_grid=params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)

best_model2 = grid.best_estimator_

In [14]:
y_pred_log4 = best_model2.predict(x_test)

y_pred_actual4 = np.expm1(y_pred_log4)
y_test_actual4 = np.expm1(y_test)

In [15]:
print("MAE:", mean_absolute_error(y_test_actual4, y_pred_actual4))
print("MSE:", mean_squared_error(y_test_actual4, y_pred_actual4))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual4, y_pred_actual4)))
print("R2 Score:", r2_score(y_test_actual4, y_pred_actual4))

MAE: 2671687.8288135594
MSE: 10798927133120.777
RMSE: 3286172.1094794744
R2 Score: 0.5468587777626848


In [16]:
# stacking hyper tuned model
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

# base models
base_models = [
    ('gbr', best_model1),
    ('xgb', best_model2)
]

# meta model
meta_model = LinearRegression()

# stacking model
stacked_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

# Train
stacked_model.fit(x_train, y_train)

# Predict
y_pred_stack = stacked_model.predict(x_test)

y_pred_stack_actual = np.expm1(y_pred_stack)
y_test_stack_actual = np.expm1(y_test)

In [17]:
print("MAE:", mean_absolute_error(y_test_stack_actual, y_pred_stack_actual))
print("MSE:", mean_squared_error(y_test_stack_actual, y_pred_stack_actual))
print("RMSE:", np.sqrt(mean_squared_error(y_test_stack_actual, y_pred_stack_actual)))
print("R2 Score:", r2_score(y_test_stack_actual, y_pred_stack_actual))

MAE: 2664275.0314983297
MSE: 10700115800391.896
RMSE: 3271103.1473177206
R2 Score: 0.5510050681794739


In [18]:
# saving model using joblib
import joblib

joblib.dump(stacked_model, 'E:/Git/blore-house-data-with-gradient-boost-and-XGBoost/xgb_gbr_stacked_model.pkl')

['E:/Git/blore-house-data-with-gradient-boost-and-XGBoost/xgb_gbr_stacked_model.pkl']